In [1]:
!pip install -q redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 5.5 MB/s eta 0:00:00


In [3]:
import redis
import json
import os
from getpass import getpass

print("Redis library imported successfully!")

Redis library imported successfully!


In [4]:
REDIS_HOST = "129.153.75.221"
REDIS_PORT = 6379
REDIS_USERNAME = "default"
REDIS_DB = 0

REDIS_PASSWORD = getpass("Enter Redis password: ")

print("Redis configuration loaded!")
print("Host:", REDIS_HOST)
print("Port:", REDIS_PORT)
print("Username:", REDIS_USERNAME)
print("Database:", REDIS_DB)

Enter Redis password: ··········
Redis configuration loaded!
Host: 129.153.75.221
Port: 6379
Username: default
Database: 0


In [5]:
try:
    redis_client = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        username=REDIS_USERNAME,
        password=REDIS_PASSWORD,
        db=REDIS_DB,
        decode_responses=True
    )

    response = redis_client.ping()

    if response:
        print("Redis Connection SUCCESS!")
        print("PING response:", response)

except Exception as e:
    print("Redis Connection FAILED!")
    print(type(e).__name__, ":", e)

Redis Connection SUCCESS!
PING response: True


In [6]:
cache_key = "faculty_burnout:FAC001"

faculty_data = {
    "Faculty_ID": "FAC001",
    "Teaching_Hours": 18,
    "Advising_Students": 8,
    "Committee_Count": 2,
    "Research_Hours": 10,
    "Admin_Hours": 4,
    "Semester_Progress": 0.65,
    "Historical_Leave_Days": 3,
    "Burnout_Risk": "Medium"
}

redis_client.set(
    cache_key,
    json.dumps(faculty_data),
    ex=3600
)

print("Data SET successfully!")
print("Cache Key:", cache_key)

Data SET successfully!
Cache Key: faculty_burnout:FAC001


In [7]:
cached_data = redis_client.get(cache_key)

if cached_data:
    cached_data = json.loads(cached_data)

    print("Data GET successfully!")
    print("Retrieved Data:")
    print(json.dumps(cached_data, indent=4))
else:
    print("No cached data found.")

Data GET successfully!
Retrieved Data:
{
    "Faculty_ID": "FAC001",
    "Teaching_Hours": 18,
    "Advising_Students": 8,
    "Committee_Count": 2,
    "Research_Hours": 10,
    "Admin_Hours": 4,
    "Semester_Progress": 0.65,
    "Historical_Leave_Days": 3,
    "Burnout_Risk": "Medium"
}


In [8]:
updated_faculty_data = faculty_data.copy()

updated_faculty_data["Teaching_Hours"] = 22
updated_faculty_data["Burnout_Risk"] = "High"

redis_client.set(
    cache_key,
    json.dumps(updated_faculty_data),
    ex=3600
)

print("Cache data UPDATED successfully!")
print(json.dumps(updated_faculty_data, indent=4))

Cache data UPDATED successfully!
{
    "Faculty_ID": "FAC001",
    "Teaching_Hours": 22,
    "Advising_Students": 8,
    "Committee_Count": 2,
    "Research_Hours": 10,
    "Admin_Hours": 4,
    "Semester_Progress": 0.65,
    "Historical_Leave_Days": 3,
    "Burnout_Risk": "High"
}


In [9]:
updated_cached_data = redis_client.get(cache_key)

if updated_cached_data:
    updated_cached_data = json.loads(updated_cached_data)

    print("Updated data retrieved successfully!")
    print(json.dumps(updated_cached_data, indent=4))
else:
    print("No cached data found.")

Updated data retrieved successfully!
{
    "Faculty_ID": "FAC001",
    "Teaching_Hours": 22,
    "Advising_Students": 8,
    "Committee_Count": 2,
    "Research_Hours": 10,
    "Admin_Hours": 4,
    "Semester_Progress": 0.65,
    "Historical_Leave_Days": 3,
    "Burnout_Risk": "High"
}


In [10]:
delete_result = redis_client.delete(cache_key)

if delete_result == 1:
    print("Cache data DELETED successfully!")
else:
    print("Cache key was not found.")

Cache data DELETED successfully!


In [11]:
remaining_data = redis_client.get(cache_key)

if remaining_data is None:
    print("DELETE verification SUCCESS!")
    print("No cached data found for:", cache_key)
else:
    print("DELETE verification FAILED!")
    print("Data still exists:", remaining_data)

DELETE verification SUCCESS!
No cached data found for: faculty_burnout:FAC001


In [12]:
print("=" * 55)
print("PHASE 4 - REDIS CACHE VERIFICATION")
print("=" * 55)

test_key = "faculty_burnout:phase4_test"

# 1. Connection
try:
    redis_client.ping()
    print("1. Redis Connection : SUCCESS")
except Exception as e:
    print("1. Redis Connection : FAILED")
    print(e)

# 2. SET
test_data = {
    "Faculty_ID": "FAC002",
    "Burnout_Risk": "Low",
    "Risk_Probability": 0.25
}

redis_client.set(
    test_key,
    json.dumps(test_data),
    ex=3600
)

print("2. SET Data         : SUCCESS")

# 3. GET
retrieved_data = redis_client.get(test_key)

if retrieved_data:
    print("3. GET Data         : SUCCESS")
else:
    print("3. GET Data         : FAILED")

# 4. UPDATE
test_data["Burnout_Risk"] = "Medium"

redis_client.set(
    test_key,
    json.dumps(test_data),
    ex=3600
)

updated_data = json.loads(redis_client.get(test_key))

if updated_data["Burnout_Risk"] == "Medium":
    print("4. UPDATE Data      : SUCCESS")
else:
    print("4. UPDATE Data      : FAILED")

# 5. DELETE
delete_result = redis_client.delete(test_key)

if delete_result == 1:
    print("5. DELETE Data      : SUCCESS")
else:
    print("5. DELETE Data      : FAILED")

# 6. Final verification
if redis_client.exists(test_key) == 0:
    print("6. DELETE Verify     : SUCCESS")
else:
    print("6. DELETE Verify     : FAILED")

print("=" * 55)
print("PHASE 4 STATUS: COMPLETED")
print("=" * 55)

PHASE 4 - REDIS CACHE VERIFICATION
1. Redis Connection : SUCCESS
2. SET Data         : SUCCESS
3. GET Data         : SUCCESS
4. UPDATE Data      : SUCCESS
5. DELETE Data      : SUCCESS
6. DELETE Verify     : SUCCESS
PHASE 4 STATUS: COMPLETED


In [13]:
print("=" * 60)
print("PHASE 4 - CACHING FRAMEWORK & REDIS")
print("=" * 60)

print("\nCompleted Operations:")
print("1. Redis Connection      : SUCCESS")
print("2. SET / Write Data      : SUCCESS")
print("3. GET / Read Data       : SUCCESS")
print("4. UPDATE Cached Data    : SUCCESS")
print("5. DELETE Cached Data    : SUCCESS")

print("\nApplication Flow:")
print("Python Application")
print("       ↓")
print("Redis Cache")
print("       ↓")
print("SET → GET → UPDATE → DELETE")

print("\nDeliverable:")
print("Working application to Redis caching flow")

print("\nPhase 4 Status: COMPLETED")
print("=" * 60)

PHASE 4 - CACHING FRAMEWORK & REDIS

Completed Operations:
1. Redis Connection      : SUCCESS
2. SET / Write Data      : SUCCESS
3. GET / Read Data       : SUCCESS
4. UPDATE Cached Data    : SUCCESS
5. DELETE Cached Data    : SUCCESS

Application Flow:
Python Application
       ↓
Redis Cache
       ↓
SET → GET → UPDATE → DELETE

Deliverable:
Working application to Redis caching flow

Phase 4 Status: COMPLETED
